## Script de predicción de precio con nueva data (cruda) llamando al API

Para este script son necesarios los siguientes ficheros:



1.   requirements.txt
2.   FuncionesTFM.py
3.   Pipeline_limpieza_data.py
4.   geo_barcelona.geojson
5.   modelo_xgb_replica.json
6.   airbnb_junio.csv

Nota: Este script fue realizado en Colab por lo que el path de lectura de ficheros incia en  "/content/", se recomienda verificar el path de ubicación si se trabaja en local.


In [1]:
# Instalar requirements con su versión
!pip install -r "/content/requirements.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.4 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=a9978f97355cfffee54588b0dfb63d0df1550214a71f80e71c6b853a560fba3f
  Stored in di

In [2]:
# Librerias
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import sys
import re
import patsy
import ast
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import torch
import torch.nn as nn
from relativeImp import relativeImp
from  ydata_profiling import ProfileReport
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import chi2_contingency
import geopandas as gpd
import pickle
import sklearn.impute as skl_imp

from fastapi import FastAPI, UploadFile, File
import io
import requests



In [3]:
# Funciones
execfile("/content/FuncionesTFM.py")
execfile("/content/Pipeline_limpieza_data.py")
execfile("/content/geo_barcelona.geojson")
geo = gpd.read_file("geo_barcelona.geojson").to_crs(25831)


In [4]:
# Leo  nuestro modelo json
from xgboost import XGBRegressor
model = XGBRegressor()
model.load_model("/content/modelo_xgb_replica.json")


In [8]:

# Definir la URL del endpoint
url = 'http://localhost:4900/predict_nueva_data_cruda'

app = FastAPI()
@app.post("/predict_nueva_data_cruda")
async def predict_csv_crudo(file: UploadFile = File(...)):
    contenido = await file.read()
    df_crudo = pd.read_csv(io.BytesIO(contenido))

    # Aplicar los mismos pipelines usados en el entrenamiento
    df_limpio = preparar_limpieza(df_crudo)
    df_id = df_limpio[['id','price_log']]
    variables_modelo = list(model.feature_names_in_)
    df_final = df_limpio[variables_modelo].astype(float)


    faltantes = [c for c in variables_modelo if c not in df_final.columns]
    if faltantes:
        return {"error": "Faltan columnas tras el preprocesamiento", "faltantes": faltantes}

    pred_log = model.predict(df_final)
    pred_precio = np.exp(pred_log)

    df_id['precio_real'] = np.exp(df_id['price_log'])

    resultado = df_final.copy()
    resultado = resultado.join(df_id[['id', 'precio_real']])
    resultado["precio_predicho_log"] = pred_log
    resultado["precio_predicho"] = pred_precio

    return resultado.to_dict(orient="records")

In [9]:

import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=4900, log_level="info")

import threading
thread = threading.Thread(target=run)

thread.start()

INFO:     Started server process [2231]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:4900 (Press CTRL+C to quit)


In [10]:
import requests

files = {"file": open("airbnb_junio.csv", "rb")}
response = requests.post("http://127.0.0.1:4900/predict_nueva_data_cruda", files=files)

resultado = response.json()
print(resultado)

Output hidden; open in https://colab.research.google.com to view.

In [11]:
#obtener las predicciones
resultado = response.json()

df_resultado = pd.DataFrame(resultado)
df_resultado  = df_resultado[['id', 'precio_real', 'precio_predicho', 'precio_predicho_log']]
df_resultado

,id,precio_real,precio_predicho,precio_predicho_log
0,18674,409.00,327.347656,5.791023
1,23197,388.00,345.537567,5.845101
2,34981,397.13,353.466187,5.867788
3,36763,43.24,39.147404,3.667334
4,40983,226.13,183.398666,5.211662
...,...,...,...,...
13350,1714453367636295798,93.87,67.174393,4.207292
13351,1714507108958061169,93.87,67.174393,4.207292
13352,1714524223290226460,183.00,356.369629,5.875968
13353,1714528494413566181,263.00,376.065125,5.929762
